# Water Potability - Preprocessing and Model Analysis

This notebook documents preprocessing, train-validation-test splitting, missing-value imputation, feature scaling, class imbalance handling, model comparison, and final model selection.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/water_potability.csv")

print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["Potability"].value_counts())

## Preprocessing Workflow

The project uses median imputation and StandardScaler. The preprocessing pipeline is fitted only on the training data to prevent data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity"
]

X = df[FEATURES]
y = df["Potability"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

## Missing Value Imputation and Feature Scaling

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_processed = pipeline.fit_transform(X_train)
X_val_processed = pipeline.transform(X_val)
X_test_processed = pipeline.transform(X_test)

print("Processed Training:", X_train_processed.shape)
print("Processed Validation:", X_val_processed.shape)
print("Processed Testing:", X_test_processed.shape)

## Class Imbalance Handling

The training scripts use `class_weight="balanced"` for Random Forest,
SVM, Logistic Regression, and Decision Tree.

Stratified splitting preserves class proportions across the datasets.

In [ ]:
print("Training Class Distribution:")
print(y_train.value_counts())

print("\nTraining Class Proportions:")
print(y_train.value_counts(normalize=True))

## Model Comparison

Five classification models are evaluated:

- Random Forest
- SVM
- Logistic Regression
- Decision Tree
- KNN

Evaluation metrics include Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
results = pd.read_csv("../outputs/validation_results.csv")
results.sort_values("ROC-AUC", ascending=False)

## Best Model

SVM achieved the highest validation ROC-AUC of **0.6570**.

In [ ]:
final_metrics = pd.read_csv("../outputs/final_test_metrics.csv")
final_metrics

## Final Test Evaluation

The selected SVM was evaluated on 492 unseen test samples.

- Accuracy: **63.41%**
- Precision: **53.03%**
- Recall: **54.69%**
- F1-Score: **53.85%**
- ROC-AUC: **0.6540**

Confusion Matrix:

`[[207, 93], [87, 105]]`

## Conclusion

The complete preprocessing and model evaluation workflow was implemented successfully.

SVM was selected based on the highest validation ROC-AUC.

The trained model and preprocessing pipeline are used by the Streamlit application.